# 03 — LLM decision engine on fixed data

Runs `TradingAgentEngine` for TEAM, MRVL and TSM at fixed candles of the fixture, with
the fixture's Daily User Prompt.

**Expected behaviour**
- the prompt's symbols and constraints are extracted in code;
- each response is parsed as strict JSON and validated (schema, authorized symbol,
  constraints) — invalid ones are logged and never accepted;
- a decision for a ticker outside the prompt is rejected whatever the model says;
- every request records tokens (provider-reported where available), latency and cost.

In MOCK mode the "model" is `tk.rule_based_reply`, a deterministic harness — its
decisions say nothing about LLM quality. REAL_LLM results are exploratory observations.

**Modes.** `MOCK` (default) makes zero external LLM calls. `REAL_LLM` needs
`QBS_AGENT_TEST_MODE=REAL_LLM` **and** `QBS_AGENT_TEST_ALLOW_PAID=1`, prints a cost
estimate first, and stops at `QBS_AGENT_TEST_MAX_CALLS` (default 20).

**Isolation.** Every run uses a fresh temporary database (`tk.workspace()`); the
production config (`var/agent/config.json`) and log (`var/agent/agent.db`) are never
read or written, and nothing here can reach `qbs.live` or a broker.

**Dependencies.** `pip install -r requirements.txt jupyter` (MOCK). REAL_LLM also needs
the selected provider's package and key (see docs/TRADING_AGENT.md).

In [1]:
import os, sys, pathlib, json, logging
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "qbs" / "trading_agent").is_dir())
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)

import pandas as pd
from qbs.trading_agent import testkit as tk

MODE = tk.test_mode()          # MOCK unless QBS_AGENT_TEST_MODE=REAL_LLM and QBS_AGENT_TEST_ALLOW_PAID=1
print("repo:", ROOT)
print("MODE:", MODE, "| paid-call cap:", tk.max_calls() if MODE == tk.REAL_LLM else 0)

repo: /home/user/qqq_boxx_strategies
MODE: MOCK | paid-call cap: 0


In [2]:
from qbs.trading_agent import session as ss, store, evaluation
from qbs.trading_agent.engine import TradingAgentEngine
from qbs.trading_agent.market_context import MarketContextBuilder
from qbs.trading_agent.prompt import parse_prompt
from qbs.trading_agent.llm import MockProvider

fx = tk.load_fixture(); day = fx.replay_date
CANDLES = [tk.et(day, 10, 15), tk.et(day, 11, 30), tk.et(day, 14, 30)]
check = tk.Check("03 LLM decision")

def fresh_engine(provider=None, mode=MODE):
    ws = tk.workspace(mode)
    clock = tk.Clock(tk.et(day, 9, 0))
    ss.save_daily_prompt(ws.cfg, fx.prompt, clock()); ss.authorize(ws.cfg, clock())
    clock.set(tk.et(day, 9, 40))
    sess, why = ss.activate(ws.cfg, clock()); assert sess, why
    sess.transition(ss.RUNNING, "nb03")
    eng = TradingAgentEngine(ws.cfg, sess, provider or tk.make_provider(mode, ws.cfg),
                             MarketContextBuilder(tk.FixtureBars(fx, clock), ws.cfg.market_context))
    return eng, clock, ws

## The Daily User Prompt, as code reads it

In [3]:
p = parse_prompt(fx.prompt, day)
check("prompt validates", p.ok, "; ".join(p.errors))
check("symbols extracted", p.symbols == ["TEAM", "MRVL", "TSM"], str(p.symbols))
check("no short selling detected", p.no_short)
check("TEAM max 4 shares, stop 196", (p.constraints["TEAM"].max_quantity, p.constraints["TEAM"].reference_stop) == (4, 196.0))
check("MRVL no-add + 9-share position", (p.constraints["MRVL"].no_add, p.constraints["MRVL"].position_qty) == (True, 9))
print(fx.prompt)
pd.DataFrame(p.constraints_dict()).T

Date: 2026-10-08

Monitor only TEAM, MRVL, and TSM.

TEAM:
Watch for a breakout above $200.
Look for breakout confirmation or a successful retest.
Maximum suggested position: 4 shares.
Reference stop loss: $196.

MRVL:
Monitor my existing 9-share position.
Evaluate whether to hold or take profit.
Do not recommend adding new shares.

TSM:
Monitor for a potential recovery after pullback.
Recovery confirmation above $302.
Maximum suggested position: 3 shares.
Do not force a trade without confirmation.

General:
Use 15-minute candles for primary analysis.
Use 30-minute and daily candles as supporting context.
Avoid unnecessary trading.
No short selling.



,max_quantity,no_add,position_qty,reference_stop,key_levels
TEAM,4,False,None,196.0,"[196.0, 200.0]"
MRVL,None,True,9,None,[]
TSM,3,False,None,None,[302.0]


## Run the engine at fixed candles

In [4]:
if MODE == tk.REAL_LLM:
    ws0 = tk.workspace(MODE)
    print("ABOUT TO SPEND:", tk.cost_preview(ws0.cfg, len(CANDLES) * 3))
eng, clock, ws = fresh_engine()
for c in CANDLES:
    clock.set(c + pd.Timedelta(minutes=1))
    eng.run_cycle(c)
dec = pd.DataFrame(store.recent_decisions(ws.cfg.db_path, 100)).sort_values(["candle_ts", "symbol"])
display(dec[["candle_ts", "symbol", "status", "action", "confidence", "entry_price",
             "stop_loss", "suggested_quantity", "reason"]])
check("one decision per symbol per candle", len(dec) == 3 * len(CANDLES))
if MODE == tk.MOCK:
    check("all MOCK decisions valid", set(dec.status) == {"VALID"}, str(set(dec.status)))

,candle_ts,symbol,status,action,confidence,entry_price,stop_loss,suggested_quantity,reason
7,2026-10-08T10:15:00-04:00,MRVL,VALID,HOLD,0.60,NaN,NaN,NaN,Position trend intact; no exit signal.
8,2026-10-08T10:15:00-04:00,TEAM,VALID,NO_TRADE,0.50,NaN,NaN,NaN,Pressing the breakout level; wait for a close ...
6,2026-10-08T10:15:00-04:00,TSM,VALID,NO_TRADE,0.50,NaN,NaN,NaN,No setup on the latest candle.
4,2026-10-08T11:30:00-04:00,MRVL,VALID,HOLD,0.60,NaN,NaN,NaN,Position trend intact; no exit signal.
5,2026-10-08T11:30:00-04:00,TEAM,VALID,BUY,0.65,200.58,196.00,4.0,15m close above 200.0 with EMA9 > EMA21 and ab...
3,2026-10-08T11:30:00-04:00,TSM,VALID,NO_TRADE,0.50,NaN,NaN,NaN,No setup on the latest candle.
1,2026-10-08T14:30:00-04:00,MRVL,VALID,SELL,0.55,NaN,NaN,3.0,Price lost VWAP and the 15m EMA9; trim.
2,2026-10-08T14:30:00-04:00,TEAM,VALID,HOLD,0.60,NaN,NaN,NaN,Breakout holding.
0,2026-10-08T14:30:00-04:00,TSM,VALID,BUY,0.65,302.26,295.96,3.0,15m close above 302.0 with EMA9 > EMA21 and ab...


## Validation catches bad output

Scripted replies: prose instead of JSON, a ticker the prompt never authorized, a short sale, and an oversized BUY. Each must be rejected and logged.

In [5]:
import re
def scripted(kind):
    def f(system, user):
        sym = re.search(r'"symbol": "([A-Z]+)"', user).group(1)
        ts = re.search(r'"candle_timestamp": "([^"]+)"', user).group(1)
        base = json.loads(tk.rule_based_reply(system, user))
        if kind == "prose":
            return "I would buy TEAM here, it looks strong."
        if kind == "unauthorized":
            base.update(symbol="NVDA")
        if kind == "short" and sym == "TSM":
            base.update(action="SELL", suggested_quantity=5, memory_update=None)
        if kind == "oversized" and sym == "TEAM":
            base.update(action="BUY", entry_price=200.5, stop_loss=196.0, suggested_quantity=10,
                        memory_update=None)
        return json.dumps(base)
    return f

rows = []
for kind, expect in [("prose", "INVALID_SCHEMA"), ("unauthorized", "INVALID_COMPLIANCE"),
                     ("short", "INVALID_COMPLIANCE"), ("oversized", "INVALID_COMPLIANCE")]:
    e, cl, w = fresh_engine(MockProvider([scripted(kind)]), mode=tk.MOCK)
    cl.set(tk.et(day, 10, 16)); rep = e.run_cycle(tk.et(day, 10, 15))
    got = {r.symbol: (r.status, "; ".join(r.errors)[:90]) for r in rep.results}
    target = {"prose": "TEAM", "unauthorized": "TEAM", "short": "TSM", "oversized": "TEAM"}[kind]
    check(f"{kind} -> {expect}", got[target][0] == expect, got[target][1])
    accepted = [r for r in store.recent_decisions(w.cfg.db_path) if r["status"] == "VALID"
                and r["symbol"] == target]
    check(f"{kind}: nothing accepted for {target}", not accepted)
    rows.append({"case": kind, "symbol": target, "status": got[target][0], "errors": got[target][1]})
pd.DataFrame(rows)

,case,symbol,status,errors
0,prose,TEAM,INVALID_SCHEMA,not valid JSON (Expecting value at char 0)
1,unauthorized,TEAM,INVALID_COMPLIANCE,symbol 'NVDA' is not the symbol requested (TEA...
2,short,TSM,INVALID_COMPLIANCE,SELL without a known long position is a short ...
3,oversized,TEAM,INVALID_COMPLIANCE,"BUY would take the position to 10 shares, over..."


## Tokens, latency and cost per request

In MOCK mode, `input_tokens` are the mock's chars/4 count (labelled `usage_source=provider` by the mock), and cost is `unavailable` because `mock-model` has no price — the point is that the columns are populated end to end. In REAL_LLM mode these are the provider's own numbers.

In [6]:
req = pd.DataFrame(store.requests_for(ws.cfg.db_path))
cols = ["symbol", "candle_ts", "model", "input_tokens", "output_tokens", "cached_input_tokens",
        "reasoning_tokens", "total_tokens", "memory_tokens_estimated", "usage_source",
        "latency_ms", "data_latency_ms", "retry_count", "total_cost", "cost_status",
        "validation_status"]
display(req[cols])
check("every request has latency", req.latency_ms.notna().all())
check("every request has a usage source", req.usage_source.notna().all())
op = evaluation.operational(ws.cfg.db_path)
summary = {k: op[k] for k in ("llm_requests", "success_rate", "validation_pass_rate",
                              "avg_latency_ms", "input_tokens", "output_tokens",
                              "estimated_cost_usd", "requests_without_cost")}
summary["mode"] = MODE
print(json.dumps(summary, indent=1, default=str))
print(tk.save_results("03_llm_decision_requests", req[cols]))

,symbol,candle_ts,model,input_tokens,output_tokens,cached_input_tokens,reasoning_tokens,total_tokens,memory_tokens_estimated,usage_source,latency_ms,data_latency_ms,retry_count,total_cost,cost_status,validation_status
0,TEAM,2026-10-08T10:15:00-04:00,mock-model,1824,128,None,None,1952,136,provider,1.0,10.8,0,None,unavailable,VALID
1,MRVL,2026-10-08T10:15:00-04:00,mock-model,1791,99,None,None,1890,109,provider,0.6,7.9,0,None,unavailable,VALID
2,TSM,2026-10-08T10:15:00-04:00,mock-model,1802,92,None,None,1894,121,provider,0.5,9.3,0,None,unavailable,VALID
3,TEAM,2026-10-08T11:30:00-04:00,mock-model,1951,141,None,None,2092,264,provider,0.5,8.1,0,None,unavailable,VALID
4,MRVL,2026-10-08T11:30:00-04:00,mock-model,1893,99,None,None,1992,210,provider,0.6,8.4,0,None,unavailable,VALID
5,TSM,2026-10-08T11:30:00-04:00,mock-model,1896,92,None,None,1988,215,provider,0.5,8.7,0,None,unavailable,VALID
6,TEAM,2026-10-08T14:30:00-04:00,mock-model,1988,91,None,None,2079,304,provider,0.5,12.3,0,None,unavailable,VALID
7,MRVL,2026-10-08T14:30:00-04:00,mock-model,1928,127,None,None,2055,250,provider,0.6,9.4,0,None,unavailable,VALID
8,TSM,2026-10-08T14:30:00-04:00,mock-model,1929,141,None,None,2070,253,provider,0.6,9.0,0,None,unavailable,VALID


{
 "llm_requests": 9,
 "success_rate": 1.0,
 "validation_pass_rate": 1.0,
 "avg_latency_ms": 0.6000000000000001,
 "input_tokens": 17002,
 "output_tokens": 1010,
 "estimated_cost_usd": null,
 "requests_without_cost": 9,
 "mode": "MOCK"
}
/tmp/claude-0/-home-user-qqq-boxx-strategies/d20be8e8-2747-505a-9165-041a5d5e0f58/scratchpad/nbres/03_llm_decision_requests_20261010T092858Z.csv


## Result

In [7]:
res = check.frame(); display(res)
print(tk.save_results("03_llm_decision_checks", res))
check.raise_if_failed()

,check,result,detail
0,prompt validates,PASS,
1,symbols extracted,PASS,"['TEAM', 'MRVL', 'TSM']"
2,no short selling detected,PASS,
3,"TEAM max 4 shares, stop 196",PASS,
4,MRVL no-add + 9-share position,PASS,
5,one decision per symbol per candle,PASS,
6,all MOCK decisions valid,PASS,{'VALID'}
7,prose -> INVALID_SCHEMA,PASS,not valid JSON (Expecting value at char 0)
8,prose: nothing accepted for TEAM,PASS,
9,unauthorized -> INVALID_COMPLIANCE,PASS,symbol 'NVDA' is not the symbol requested (TEA...


/tmp/claude-0/-home-user-qqq-boxx-strategies/d20be8e8-2747-505a-9165-041a5d5e0f58/scratchpad/nbres/03_llm_decision_checks_20261010T092858Z.csv


**Troubleshooting**
- *REAL_LLM: INVALID_SCHEMA*: read `error` in the requests table; set
  `log_full_prompts=True` on the workspace config to keep the raw reply.
- *REAL_LLM: LLM_ERROR*: missing key or package, wrong model id, or rate limit — the
  error column names which.
- *Cost unavailable*: add the model to a pricing file (`QBS_AGENT_PRICING_FILE`).